# Setting up the environment

In [ ]:
!pip install -q condacolab
import condacolab
condacolab.install()

✨🍰✨ Everything looks OK!


In [ ]:
# Use mamba for speed/stability
!mamba create -n micro_sam3 -y -c conda-forge python=3.12
!source /usr/local/bin/activate && conda activate micro_sam3 && \
  mamba install -y -c conda-forge vigra pooch python-xxhash python-elf

# Install PyTorch (CUDA 12.4 wheels)
!source /usr/local/bin/activate && conda activate micro_sam3 && \
  pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124

# Pip deps
!source /usr/local/bin/activate && conda activate micro_sam3 && \
  pip install -q "napari[all]" zarr==2.18.4 segment-anything kornia


In [ ]:
# Clone MicroSAM
!source /usr/local/bin/activate && conda activate micro_sam3 && \
  cd /content && \
  git clone https://github.com/computational-cell-analytics/micro-sam.git

# Install MicroSAM
!source /usr/local/bin/activate && conda activate micro_sam3 && \
  cd /content/micro-sam && \
  pip install -e .

# Clone torch-em
!source /usr/local/bin/activate && conda activate micro_sam3 && \
  cd /content && \
  git clone https://github.com/constantinpape/torch-em.git

# Install torch-em
!source /usr/local/bin/activate && conda activate micro_sam3 && \
  cd /content/torch-em && \
  pip install -e .


fatal: destination path 'micro-sam' already exists and is not an empty directory.
Obtaining file:///content/micro-sam
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for micro-sam (pyproject.toml) ... done
  Created wheel for micro-sam: filename=micro_sam-1.7.5-0.editable-py3-none-any.whl size=4748 sha256=96f31bd3541a8ebfce7758dc50503d016d3a603b768aced102e808b989e5c3ec
  Stored in directory: /tmp/pip-ephem-wheel-cache-wrthkaqq/wheels/a7/57/78/cc04d17597bc17d7717d8b09b6543062b4028f6757cc83fa6b
Successfully built micro-sam
fatal: destination path 'torch-em' already exists and is not an empty directory.
Obtaining file:///content/torch-em
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable meta

In [ ]:
%%bash
source /usr/local/bin/activate
conda activate micro_sam3

python - << 'EOF'
import warnings
warnings.filterwarnings("ignore")

import torch
print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

import micro_sam
print("MicroSAM imported successfully")

import torch_em
print("torch-em imported successfully")
EOF


Torch version: 2.6.0+cu124
CUDA available: True
MicroSAM imported successfully
torch-em imported successfully


In [ ]:
%%bash
source /usr/local/bin/activate
conda activate micro_sam3

pip uninstall -y numcodecs zarr
pip install -q "numcodecs==0.12.1" "zarr==2.18.4"
python - << 'EOF'
import zarr, numcodecs
print("zarr:", zarr.__version__)
print("numcodecs:", numcodecs.__version__)
from numcodecs.blosc import cbuffer_sizes
print("cbuffer_sizes import OK")
EOF


Found existing installation: numcodecs 0.16.5
Uninstalling numcodecs-0.16.5:
  Successfully uninstalled numcodecs-0.16.5
Found existing installation: zarr 2.18.4
Uninstalling zarr-2.18.4:
  Successfully uninstalled zarr-2.18.4
zarr: 2.18.4
numcodecs: 0.12.1
cbuffer_sizes import OK


In [ ]:
%%bash
source /usr/local/bin/activate
conda activate micro_sam3

pip uninstall -y numcodecs zarr
pip install -q "numcodecs==0.12.1" "zarr==2.18.4"
python - << 'EOF'
import zarr, numcodecs
print("zarr:", zarr.__version__)
print("numcodecs:", numcodecs.__version__)
from numcodecs.blosc import cbuffer_sizes
print("cbuffer_sizes import OK")
EOF


Found existing installation: numcodecs 0.12.1
Uninstalling numcodecs-0.12.1:
  Successfully uninstalled numcodecs-0.12.1
Found existing installation: zarr 2.18.4
Uninstalling zarr-2.18.4:
  Successfully uninstalled zarr-2.18.4
zarr: 2.18.4
numcodecs: 0.12.1
cbuffer_sizes import OK


In [ ]:
%%bash
source /usr/local/bin/activate
conda activate micro_sam3

pip install -q tensorboard
python - << 'EOF'
import tensorboard
print("tensorboard:", tensorboard.__version__)
from torch.utils.tensorboard import SummaryWriter
print("SummaryWriter import OK")
EOF


tensorboard: 2.20.0
SummaryWriter import OK


In [ ]:
%%bash
source /usr/local/bin/activate
conda activate micro_sam3

# safest: conda-forge (brings compatible deps)
mamba install -y -c conda-forge xarray netcdf4

python - << 'EOF'
import xarray as xr
print("xarray:", xr.__version__)
print("xarray import OK")
EOF


In [ ]:
%%bash
source /usr/local/bin/activate
conda activate micro_sam3

mamba install -y -c conda-forge natsort

python - << 'EOF'
import warnings
warnings.filterwarnings("ignore")

from glob import glob
from IPython.display import FileLink

import numpy as np
import imageio.v3 as imageio
from matplotlib import pyplot as plt
from skimage.measure import label as connected_components


import torch

from torch_em.util.debug import check_loader
from torch_em.data import MinInstanceSampler
from torch_em.util.util import get_random_colors

import micro_sam.training as sam_training
from micro_sam.sample_data import (
    fetch_tracking_example_data,
    fetch_tracking_segmentation_data
)
from micro_sam.automatic_segmentation import (
    get_predictor_and_segmenter,
    automatic_instance_segmentation
)

print("All imports successful")
EOF

In [ ]:
from google.colab import drive
drive.mount("/content/drive")


Mounted at /content/drive


# LOADING DATASET AND AUGMENTATION

In [ ]:
%%bash
set -e
source /usr/local/bin/activate
conda activate micro_sam3

cat > /content/obtain_lucchi_urocell_kasthuri_vnc_em_datasets.py <<'PY'
import os
import random
import numpy as np
import torch
from torch.utils.data import random_split

from torch_em import get_data_loader
from torch_em.data import ConcatDataset, MinInstanceSampler, datasets
from micro_sam.training.util import ResizeRawTrafo, ResizeLabelTrafo


# -----------------------------
# helpers
# -----------------------------
def _to_tt_split(split: str) -> str:
    return "test" if split in ("val", "valid", "validation") else split


def _split_dataset(ds, val_fraction=0.1, seed=0):
    n_total = len(ds)
    n_val = max(1, int(val_fraction * n_total))
    n_train = n_total - n_val
    gen = torch.Generator().manual_seed(seed)
    return random_split(ds, [n_train, n_val], generator=gen)


def _common_transforms(patch_shape):
    raw_tf = ResizeRawTrafo(patch_shape[1:], do_rescaling=False, ensure_rgb=False)
    lab_tf = ResizeLabelTrafo(patch_shape[1:])
    return raw_tf, lab_tf


def _set_sampling_attempts(ds, n=5000):
    if hasattr(ds, "max_sampling_attempts"):
        ds.max_sampling_attempts = n
    if hasattr(ds, "dataset") and hasattr(ds.dataset, "max_sampling_attempts"):
        ds.dataset.max_sampling_attempts = n


# -----------------------------
# augmentation helpers
# -----------------------------
def _to_numpy(x):
    if isinstance(x, torch.Tensor):
        return x.detach().cpu().numpy(), True, x.dtype
    return np.asarray(x), False, None


def _from_numpy(x, was_tensor, dtype):
    x = np.ascontiguousarray(x)
    if was_tensor:
        return torch.from_numpy(x).to(dtype=dtype)
    return x


def augment_em(raw, label):
    raw_np, raw_was_tensor, raw_dtype = _to_numpy(raw)
    lab_np, lab_was_tensor, lab_dtype = _to_numpy(label)

    if random.random() < 0.5:
        raw_np = np.flip(raw_np, axis=-1)
        lab_np = np.flip(lab_np, axis=-1)

    if random.random() < 0.5:
        raw_np = np.flip(raw_np, axis=-2)
        lab_np = np.flip(lab_np, axis=-2)

    k = random.randint(0, 3)
    if k > 0:
        raw_np = np.rot90(raw_np, k, axes=(-2, -1))
        lab_np = np.rot90(lab_np, k, axes=(-2, -1))

    if random.random() < 0.3:
        factor = 0.9 + 0.2 * random.random()
        raw_np = raw_np * factor

    if random.random() < 0.3:
        bias = np.random.uniform(-0.05, 0.05)
        raw_np = raw_np + bias

    if random.random() < 0.3:
        sigma = np.random.uniform(0.005, 0.02)
        raw_np = raw_np + np.random.normal(0.0, sigma, size=raw_np.shape)

    raw_out = _from_numpy(raw_np, raw_was_tensor, raw_dtype)
    lab_out = _from_numpy(lab_np, lab_was_tensor, lab_dtype)
    return raw_out, lab_out


class AugmentedUpsampledDataset(torch.utils.data.Dataset):
    """
    Upsample a dataset to target_size by cycling through samples and applying augmentation.
    Important: forward torch-em dataset attributes needed by ConcatDataset.
    """
    def __init__(self, dataset, target_size):
        self.dataset = dataset
        self.target_size = int(target_size)

        # forward common dataset attributes used by torch-em / ConcatDataset
        self.ndim = getattr(dataset, "ndim", None)
        self.raw_channels = getattr(dataset, "raw_channels", None)
        self.n_samples = self.target_size

        # keep max_sampling_attempts if present
        if hasattr(dataset, "max_sampling_attempts"):
            self.max_sampling_attempts = dataset.max_sampling_attempts

    def __len__(self):
        return self.target_size

    def __getitem__(self, idx):
        base_idx = idx % len(self.dataset)
        sample = self.dataset[base_idx]

        if isinstance(sample, dict):
            sample = dict(sample)
            raw_key = "raw" if "raw" in sample else None
            label_key = "label" if "label" in sample else ("labels" if "labels" in sample else None)
            if raw_key is None or label_key is None:
                raise KeyError(f"Expected sample dict to contain 'raw' and 'label'/'labels', got keys: {list(sample.keys())}")

            raw_aug, label_aug = augment_em(sample[raw_key], sample[label_key])
            sample[raw_key] = raw_aug
            sample[label_key] = label_aug
            return sample

        if isinstance(sample, (tuple, list)) and len(sample) >= 2:
            raw_aug, label_aug = augment_em(sample[0], sample[1])
            rest = list(sample[2:])
            return type(sample)((raw_aug, label_aug, *rest)) if isinstance(sample, tuple) else [raw_aug, label_aug, *rest]

        raise TypeError(f"Unsupported sample type from dataset: {type(sample)}")


# -----------------------------
# dataset getters
# -----------------------------
def _get_lucchi(input_path, patch_shape, split):
    split = _to_tt_split(split)
    path = os.path.join(input_path, "lucchi")
    raw_tf, lab_tf = _common_transforms(patch_shape)
    return datasets.get_lucchi_dataset(
        path=path,
        patch_shape=patch_shape,
        download=True,
        split=split,
        ndim=2,
        sampler=MinInstanceSampler(min_num_instances=2),
        raw_transform=raw_tf,
        label_transform=lab_tf,
    )


def _get_urocell_full(input_path, patch_shape, target="mito"):
    path = os.path.join(input_path, "urocell")
    raw_tf, lab_tf = _common_transforms(patch_shape)
    return datasets.get_uro_cell_dataset(
        path=path,
        target=target,
        patch_shape=patch_shape,
        download=True,
        ndim=2,
        sampler=MinInstanceSampler(min_num_instances=2),
        raw_transform=raw_tf,
        label_transform=lab_tf,
    )


def _get_kasthuri(input_path, patch_shape, split):
    split = _to_tt_split(split)
    path = os.path.join(input_path, "kasthuri")
    raw_tf, lab_tf = _common_transforms(patch_shape)
    return datasets.get_kasthuri_dataset(
        path=path,
        split=split,
        patch_shape=patch_shape,
        download=True,
        ndim=2,
        sampler=MinInstanceSampler(min_num_instances=2),
        raw_transform=raw_tf,
        label_transform=lab_tf,
    )


def _get_vnc_try_split(input_path, patch_shape, split):
    split = _to_tt_split(split)
    path = os.path.join(input_path, "vnc")
    raw_tf, lab_tf = _common_transforms(patch_shape)
    return datasets.get_vnc_mito_dataset(
        path=path,
        split=split,
        patch_shape=patch_shape,
        download=True,
        ndim=2,
        sampler=MinInstanceSampler(min_num_instances=2),
        raw_transform=raw_tf,
        label_transform=lab_tf,
    )


def _get_vnc_train_val(input_path, patch_shape, val_fraction=0.1, seed=0):
    try:
        vnc_train = _get_vnc_try_split(input_path, patch_shape, split="train")
        vnc_val = _get_vnc_try_split(input_path, patch_shape, split="val")
        return vnc_train, vnc_val
    except TypeError:
        path = os.path.join(input_path, "vnc")
        raw_tf, lab_tf = _common_transforms(patch_shape)
        vnc_full = datasets.get_vnc_mito_dataset(
            path=path,
            patch_shape=patch_shape,
            download=True,
            ndim=2,
            sampler=MinInstanceSampler(min_num_instances=2),
            raw_transform=raw_tf,
            label_transform=lab_tf,
        )
        vnc_train, vnc_val = _split_dataset(vnc_full, val_fraction=val_fraction, seed=seed)
        return vnc_train, vnc_val


# -----------------------------
# TRAIN LOADERS (generalist)
# -----------------------------
def get_generalist_lucchi_urocell_kasthuri_vnc_loaders(
    input_path,
    patch_shape,
    num_workers=0,
    uro_target="mito",
    uro_val_fraction=0.1,
    vnc_val_fraction=0.1,
    split_seed=0,
    batch_size=1,
    augment_to_max_size=True,
):
    lucchi_train = _get_lucchi(input_path, patch_shape, split="train")
    lucchi_val = _get_lucchi(input_path, patch_shape, split="val")

    uro_full = _get_urocell_full(input_path, patch_shape, target=uro_target)
    uro_train, uro_val = _split_dataset(uro_full, val_fraction=uro_val_fraction, seed=split_seed)

    kas_train = _get_kasthuri(input_path, patch_shape, split="train")
    kas_val = _get_kasthuri(input_path, patch_shape, split="val")

    vnc_train, vnc_val = _get_vnc_train_val(
        input_path, patch_shape, val_fraction=vnc_val_fraction, seed=split_seed
    )

    for ds in [lucchi_train, lucchi_val, kas_train, kas_val, uro_full, vnc_train, vnc_val]:
        _set_sampling_attempts(ds, 5000)

    if augment_to_max_size:
        target_size = max(len(lucchi_train), len(uro_train), len(kas_train), len(vnc_train))
        if len(lucchi_train) < target_size:
            lucchi_train = AugmentedUpsampledDataset(lucchi_train, target_size)
        if len(uro_train) < target_size:
            uro_train = AugmentedUpsampledDataset(uro_train, target_size)
        if len(kas_train) < target_size:
            kas_train = AugmentedUpsampledDataset(kas_train, target_size)
        if len(vnc_train) < target_size:
            vnc_train = AugmentedUpsampledDataset(vnc_train, target_size)

    train_ds = ConcatDataset(lucchi_train, uro_train, kas_train, vnc_train)
    val_ds = ConcatDataset(lucchi_val, uro_val, kas_val, vnc_val)

    train_loader = get_data_loader(
        train_ds,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
    )
    val_loader = get_data_loader(
        val_ds,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
    )
    return train_loader, val_loader


# -----------------------------
# SPECIALIST LOADERS
# -----------------------------
def get_specialist_loaders(
    dataset,
    input_path,
    patch_shape,
    num_workers=0,
    uro_target="mito",
    uro_val_fraction=0.1,
    vnc_val_fraction=0.1,
    split_seed=0,
    batch_size=1,
):
    if dataset == "lucchi":
        train_ds = _get_lucchi(input_path, patch_shape, split="train")
        val_ds = _get_lucchi(input_path, patch_shape, split="val")

    elif dataset == "urocell":
        full = _get_urocell_full(input_path, patch_shape, target=uro_target)
        train_ds, val_ds = _split_dataset(full, val_fraction=uro_val_fraction, seed=split_seed)

    elif dataset == "kasthuri":
        train_ds = _get_kasthuri(input_path, patch_shape, split="train")
        val_ds = _get_kasthuri(input_path, patch_shape, split="val")

    elif dataset == "vnc":
        train_ds, val_ds = _get_vnc_train_val(
            input_path, patch_shape, val_fraction=vnc_val_fraction, seed=split_seed
        )
    else:
        raise ValueError("dataset must be one of: lucchi, urocell, kasthuri, vnc")

    _set_sampling_attempts(train_ds, 5000)
    _set_sampling_attempts(val_ds, 5000)

    train_loader = get_data_loader(train_ds, batch_size=batch_size, shuffle=True, num_workers=num_workers)
    val_loader = get_data_loader(val_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers)
    return train_loader, val_loader


# -----------------------------
# EVAL LOADERS
# -----------------------------
def get_eval_loaders(
    input_path,
    patch_shape,
    num_workers=0,
    uro_target="mito",
    uro_val_fraction=0.1,
    vnc_val_fraction=0.1,
    split_seed=0,
    batch_size=1,
):
    lucchi_test = _get_lucchi(input_path, patch_shape, split="test")
    _set_sampling_attempts(lucchi_test, 5000)

    uro_full = _get_urocell_full(input_path, patch_shape, target=uro_target)
    _, uro_val = _split_dataset(uro_full, val_fraction=uro_val_fraction, seed=split_seed)
    _set_sampling_attempts(uro_full, 5000)

    kas_test = _get_kasthuri(input_path, patch_shape, split="test")
    _set_sampling_attempts(kas_test, 5000)

    vnc_train, vnc_val = _get_vnc_train_val(
        input_path, patch_shape, val_fraction=vnc_val_fraction, seed=split_seed
    )
    _set_sampling_attempts(vnc_train, 5000)
    _set_sampling_attempts(vnc_val, 5000)

    lucchi_test_loader = get_data_loader(lucchi_test, batch_size=batch_size, shuffle=False, num_workers=num_workers)
    urocell_val_loader = get_data_loader(uro_val, batch_size=batch_size, shuffle=False, num_workers=num_workers)
    kasthuri_test_loader = get_data_loader(kas_test, batch_size=batch_size, shuffle=False, num_workers=num_workers)
    vnc_val_loader = get_data_loader(vnc_val, batch_size=batch_size, shuffle=False, num_workers=num_workers)

    return lucchi_test_loader, urocell_val_loader, kasthuri_test_loader, vnc_val_loader
PY

In [ ]:
%%bash
set -e
source /usr/local/bin/activate
conda activate micro_sam3

python - <<'PY'
from obtain_lucchi_urocell_kasthuri_vnc_em_datasets import (
    get_generalist_lucchi_urocell_kasthuri_vnc_loaders
)

train_loader, val_loader = get_generalist_lucchi_urocell_kasthuri_vnc_loaders(
    input_path="/content/em_data",
    patch_shape=(1, 512, 512),
    uro_target="mito",
    uro_val_fraction=0.1,
    vnc_val_fraction=0.1,
    split_seed=0,
    batch_size=1,
    num_workers=0,
    augment_to_max_size=True   # ← added augmentation balancing
)

print("Sanity check passed")
print("Train batches:", len(train_loader))
print("Val batches  :", len(val_loader))

batch = next(iter(train_loader))
if isinstance(batch, dict):
    print("Batch keys:", batch.keys())
else:
    print("Batch type:", type(batch))
PY

Download successful and checksums agree.
Download successful and checksums agree.
Download successful and checksums agree.
Download successful and checksums agree.
Sanity check passed
Train batches: 2372
Val batches  : 1301
Batch type: <class 'list'>


Download http://www.casser.io/files/lucchi_pp.zip to /content/em_data/lucchi/lucchi.zip: 100%|██████████| 216M/216M [00:09<00:00, 24.4MB/s]
Load volume: 100%|██████████| 164/164 [00:00<00:00, 804.39it/s]
Download https://github.com/MancaZerovnikMekuc/UroCell/archive/refs/heads/master.zip to /content/em_data/urocell/uro_cell.zip (unknown file size): 153MB [00:06, 23.3MB/s]
Download http://www.casser.io/files/kasthuri_pp.zip  to /content/em_data/kasthuri/kasthuri.zip: 100%|██████████| 150M/150M [00:06<00:00, 22.5MB/s]
Load volume: 100%|██████████| 84/84 [00:00<00:00, 187.16it/s]
Download https://github.com/unidesigner/groundtruth-drosophila-vnc/archive/refs/heads/master.zip to /content/em_data/vnc/vnc.zip (unknown file size): 40.7MB [00:02, 19.5MB/s]


# TRAINING

In [ ]:
%%bash
set -e
source /usr/local/bin/activate
conda activate micro_sam3

cat > /content/finetune_mito_nuc_em_generalist.py <<'PY'
import os
import argparse
import time
import torch
import torch_em

from micro_sam.util import export_custom_sam_model, get_device
from micro_sam.training.util import get_trainable_sam_model, ConvertToSamInputs
from micro_sam.training import joint_sam_trainer as joint_trainers

from obtain_lucchi_urocell_kasthuri_vnc_em_datasets import (
    get_generalist_lucchi_urocell_kasthuri_vnc_loaders
)


# -----------------------------
# LLRD helpers (LoRA-only)
# -----------------------------
def _freeze_block_params(block):
    for p in block.parameters():
        p.requires_grad = False


def _collect_block_lora_params(block):
    """
    Collect LoRA params from micro-sam ViT block LoRA adapters.
    """
    params = []
    attn = getattr(block, "attn", None)
    qkv = getattr(attn, "qkv", None) if attn is not None else None
    if qkv is None:
        return params

    for attr in ("w_a_linear_q", "w_b_linear_q", "w_a_linear_v", "w_b_linear_v"):
        m = getattr(qkv, attr, None)
        if m is not None:
            params.extend([p for p in m.parameters() if p.requires_grad])
    return params


def build_llrd_groups_lora_only(model, base_lr: float, layer_decay: float, start_block: int, lr_other: float):
    """
    Param groups:
      - blocks < start_block: frozen
      - LoRA params in blocks >= start_block: LLRD
      - everything else trainable: lr_other
    """
    assert hasattr(model.sam.image_encoder, "blocks"), "Expected ViT-style encoder with .blocks"
    blocks = model.sam.image_encoder.blocks
    n_blocks = len(blocks)  # ViT-B -> 12

    for i in range(0, min(start_block, n_blocks)):
        _freeze_block_params(blocks[i])

    param_groups = []
    lora_ids = set()

    for i in range(start_block, n_blocks):
        lr_i = base_lr * (layer_decay ** (n_blocks - i - 1))
        lora_params = _collect_block_lora_params(blocks[i])
        if lora_params:
            param_groups.append({"params": lora_params, "lr": lr_i})
            for p in lora_params:
                lora_ids.add(id(p))

    other = [p for p in model.parameters() if p.requires_grad and id(p) not in lora_ids]
    if other:
        param_groups.append({"params": other, "lr": lr_other})

    if not param_groups:
        raise RuntimeError("No optimizer param groups built. LoRA not found / not trainable.")

    return param_groups


# -----------------------------
# Training (LoRA + LLRD)
# -----------------------------
def train_sam_lora_llrd(
    name,
    model_type,
    train_loader,
    val_loader,
    n_objects_per_batch,
    n_sub_iteration,
    checkpoint_path,
    device,
    lr,
    n_iterations,
    save_root,
    scheduler_kwargs,
    box_distortion_factor,
    peft_kwargs,
    layer_decay,
    lora_start_block,
    lr_other_factor,
    lr_unetr_factor,
    weight_decay=1e-4,
):
    device = get_device(device)

    model, state = get_trainable_sam_model(
        model_type=model_type,
        device=device,
        freeze=None,
        checkpoint_path=checkpoint_path,
        return_state=True,
        peft_kwargs=peft_kwargs,
    )

    n_trainable = sum(p.requires_grad for p in model.parameters())
    n_lora_named = sum(p.requires_grad and ("lora" in n.lower()) for n, p in model.named_parameters())
    print("Trainable params:", n_trainable, "| LoRA-named trainables:", n_lora_named)

    convert_inputs = ConvertToSamInputs(
        transform=model.transform,
        box_distortion_factor=box_distortion_factor
    )

    from micro_sam.instance_segmentation import get_unetr
    unetr = get_unetr(
        image_encoder=model.sam.image_encoder,
        decoder_state=state.get("decoder_state", None),
        device=device,
    )

    lr_other = lr * lr_other_factor
    lr_unetr = lr * lr_unetr_factor

    param_groups = build_llrd_groups_lora_only(
        model=model,
        base_lr=lr,
        layer_decay=layer_decay,
        start_block=lora_start_block,
        lr_other=lr_other,
    )

    # UNETR decoder params at smaller LR (exclude encoder params)
    unetr_decoder_params = []
    for pname, p in unetr.named_parameters():
        if p.requires_grad and not pname.startswith("encoder"):
            unetr_decoder_params.append(p)
    if unetr_decoder_params:
        param_groups.append({"params": unetr_decoder_params, "lr": lr_unetr})

    group_lrs = [g["lr"] for g in param_groups]
    print("Optimizer groups:", len(param_groups))
    print("Group LRs:", group_lrs)
    print("Group LRs (min..max):", min(group_lrs), "..", max(group_lrs))
    print("LR other factor:", lr_other_factor, "=>", lr_other)
    print("LR unetr factor:", lr_unetr_factor, "=>", lr_unetr)

    optimizer = torch.optim.AdamW(param_groups, weight_decay=weight_decay)

    if scheduler_kwargs is None:
        scheduler_kwargs = {"mode": "min", "factor": 0.9, "patience": 15}
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer=optimizer, **scheduler_kwargs)

    instance_seg_loss = torch_em.loss.DiceBasedDistanceLoss(mask_distances_in_bg=True)

    trainer = joint_trainers.JointSamTrainer(
        name=name,
        save_root=save_root,
        train_loader=train_loader,
        val_loader=val_loader,
        model=model,
        optimizer=optimizer,
        device=device,
        lr_scheduler=scheduler,
        logger=joint_trainers.JointSamLogger,
        log_image_interval=100,
        mixed_precision=True,
        convert_inputs=convert_inputs,
        n_objects_per_batch=n_objects_per_batch,
        n_sub_iteration=n_sub_iteration,
        compile_model=False,
        unetr=unetr,
        instance_loss=instance_seg_loss,
        instance_metric=instance_seg_loss,
        early_stopping=None,
        mask_prob=0.5,
    )

    start_time = time.time()
    trainer.fit(iterations=n_iterations, overwrite_training=True)
    end_time = time.time()
    print(f"Total training time: {(end_time - start_time)/60:.2f} minutes ({(end_time - start_time)/3600:.2f} hours)")


# -----------------------------
# Main entry: Generalist training
# -----------------------------
def finetune_mito_nuc_em_generalist(args):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model_type = args.model_type

    checkpoint_path = args.checkpoint_path
    if checkpoint_path in (None, "", "none", "None"):
        checkpoint_path = None
    if checkpoint_path is not None and not os.path.isfile(checkpoint_path):
        raise FileNotFoundError(f"Checkpoint not found: {checkpoint_path}")

    patch_shape = (1, 384, 384)
    n_objects_per_batch = args.n_objects

    checkpoint_name = (
        f"{args.model_type}/mito_nuc_em_generalist_sam_lora_llrd_r{args.lora_rank}"
        f"_d{args.layer_decay}_lr{args.lr}_of{args.lr_other_factor}_uf{args.lr_unetr_factor}"
        f"_aug{int(args.augment_to_max_size)}"
    )

    train_loader, val_loader = get_generalist_lucchi_urocell_kasthuri_vnc_loaders(
        input_path=args.input_path,
        patch_shape=patch_shape,
        num_workers=args.num_workers,
        uro_target="mito",
        uro_val_fraction=args.uro_val_fraction,
        vnc_val_fraction=args.vnc_val_fraction,
        split_seed=args.split_seed,
        batch_size=1,
        augment_to_max_size=args.augment_to_max_size,
    )

    scheduler_kwargs = {"mode": "min", "factor": 0.9, "patience": 15}

    peft_kwargs = {
        "rank": args.lora_rank,
        "update_matrices": ["q", "v"],
        "attention_layers_to_update": list(range(args.lora_start_block, 12)),
    }

    print("Starting generalist training (LoRA + LLRD)")
    print("Model type:", model_type)
    print("Checkpoint path:", checkpoint_path)
    print("Iterations:", args.iterations)
    print("Objects per batch:", n_objects_per_batch)
    print("Device:", device)
    print("Decoder enabled:", True)
    print("Augment to max size:", args.augment_to_max_size)
    print("LoRA rank:", args.lora_rank)
    print("LoRA blocks:", peft_kwargs["attention_layers_to_update"])
    print("LoRA matrices:", peft_kwargs["update_matrices"])
    print("Top LR:", args.lr)
    print("LLRD:", args.layer_decay)
    print("Other LR factor:", args.lr_other_factor)
    print("UNETR LR factor:", args.lr_unetr_factor)

    train_sam_lora_llrd(
        name=checkpoint_name,
        model_type=model_type,
        train_loader=train_loader,
        val_loader=val_loader,
        n_objects_per_batch=n_objects_per_batch,
        n_sub_iteration=4,
        checkpoint_path=checkpoint_path,
        device=device,
        lr=args.lr,
        n_iterations=args.iterations,
        save_root=args.save_root,
        scheduler_kwargs=scheduler_kwargs,
        box_distortion_factor=0.05,
        peft_kwargs=peft_kwargs,
        layer_decay=args.layer_decay,
        lora_start_block=args.lora_start_block,
        lr_other_factor=args.lr_other_factor,
        lr_unetr_factor=args.lr_unetr_factor,
        weight_decay=1e-4,
    )

    if args.export_path is not None:
        best_ckpt = os.path.join(
            "" if args.save_root is None else args.save_root,
            "checkpoints", checkpoint_name, "best.pt"
        )
        export_custom_sam_model(
            checkpoint_path=best_ckpt,
            model_type=model_type,
            save_path=args.export_path,
        )


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--input_path", "-i", default="/content/em_data")
    parser.add_argument("--model_type", "-m", default="vit_b_em_organelles")
    parser.add_argument("--save_root", "-s", default="/content/microsam_runs")
    parser.add_argument("--iterations", type=int, default=20000)
    parser.add_argument("--export_path", "-e", default=None)
    parser.add_argument("--n_objects", type=int, default=15)
    parser.add_argument("--num_workers", type=int, default=2)

    parser.add_argument("--uro_val_fraction", type=float, default=0.1)
    parser.add_argument("--vnc_val_fraction", type=float, default=0.1)
    parser.add_argument("--split_seed", type=int, default=0)

    parser.add_argument("--checkpoint_path", "-c", default=None)

    # LoRA knobs
    parser.add_argument("--lora_rank", type=int, default=8)
    parser.add_argument("--lora_start_block", type=int, default=6)

    # LR knobs
    parser.add_argument("--lr", type=float, default=1e-4)
    parser.add_argument("--layer_decay", type=float, default=0.95)
    parser.add_argument("--lr_other_factor", type=float, default=0.1)
    parser.add_argument("--lr_unetr_factor", type=float, default=0.1)

    # Augmentation-based equalization
    parser.add_argument(
        "--augment_to_max_size",
        action="store_true",
        help="Augment and upsample smaller training datasets to the largest dataset size"
    )

    args = parser.parse_args()
    finetune_mito_nuc_em_generalist(args)


if __name__ == "__main__":
    main()
PY

In [ ]:
%%bash
tensorboard --logdir /content/microsam_runs --port 6006 --bind_all > /content/tb.log 2>&1 &
echo "TensorBoard running on port 6006"


TensorBoard running on port 6006


In [ ]:
from google.colab import output
url = output.eval_js("google.colab.kernel.proxyPort(6006)")
print(url)


https://6006-gpu-l4-s-kkb-usw4a2-2g1i948qklc2t-a.us-west4-2.prod.colab.dev


In [ ]:
%%bash
set -e
source /usr/local/bin/activate
conda activate micro_sam3

python /content/finetune_mito_nuc_em_generalist.py \
  --iterations 20000 \
  --lora_rank 8 \
  --lora_start_block 6 \
  --lr 1e-4 \
  --layer_decay 0.95 \
  --lr_other_factor 0.1 \
  --lr_unetr_factor 0.1 \
  --augment_to_max_size

In [ ]:
!mkdir -p /content/drive/MyDrive/lora
!cp -r /content/microsam_runs /content/drive/MyDrive/lora/


In [ ]:
%%bash
set -e
source /usr/local/bin/activate
conda activate micro_sam3

python - << 'PY'
import os
import torch
from micro_sam.training.util import get_trainable_sam_model

MODEL_TYPE = "vit_b_em_organelles"
RANK = 8
LORA_START = 6
LORA_LAYERS = list(range(LORA_START, 12))
UPDATE = ["q", "v"]

BEST_PT = "/content/microsam_runs/checkpoints/vit_b_em_organelles/mito_nuc_em_generalist_sam_lora_llrd_r8_d0.95_lr0.0001_of0.1_uf0.1_aug1/best.pt"
OUT_PT  = "/content/microsam_runs/checkpoints/vit_b_em_organelles/mito_nuc_em_generalist_sam_lora_llrd_r8_d0.95_lr0.0001_of0.1_uf0.1_aug1/mergedbest.pt"

ckpt = torch.load(BEST_PT, map_location="cpu", weights_only=False)
assert isinstance(ckpt, dict), "Expected micro-sam checkpoint dict."

# Locate model weights in checkpoint
if "model_state" in ckpt:
    model_key = "model_state"
elif "state_dict" in ckpt:
    model_key = "state_dict"
else:
    raise RuntimeError(f"Cannot find model weights key. Keys: {list(ckpt.keys())}")

# Sanity: ensure decoder state exists in ckpt (any key containing 'decoder')
decoder_keys = [k for k in ckpt.keys() if "decoder" in k.lower()]
if not decoder_keys:
    raise RuntimeError(f"No decoder state found in ckpt keys: {list(ckpt.keys())}")
print("Decoder-related keys found:", decoder_keys)

# Build TrainableSAM with same LoRA config (CPU)
trainable = get_trainable_sam_model(
    model_type=MODEL_TYPE,
    device="cpu",
    peft_kwargs={
        "rank": RANK,
        "update_matrices": UPDATE,
        "attention_layers_to_update": LORA_LAYERS,
    },
)

trainable.load_state_dict(ckpt[model_key], strict=True)
sam = trainable.sam  # underlying SAM

# Merge LoRA into qkv for chosen layers (Q and V only)
for i, blk in enumerate(sam.image_encoder.blocks):
    if i not in LORA_LAYERS:
        continue
    qkv = blk.attn.qkv
    base = qkv.qkv_proj
    W = base.weight.data
    d = base.in_features

    if hasattr(qkv, "w_a_linear_q"):
        W[0:d, :] += (qkv.w_b_linear_q.weight @ qkv.w_a_linear_q.weight) * qkv.alpha
    if hasattr(qkv, "w_a_linear_v"):
        W[2*d:3*d, :] += (qkv.w_b_linear_v.weight @ qkv.w_a_linear_v.weight) * qkv.alpha

    blk.attn.qkv = base  # restore plain Linear

# Write new checkpoint: keep everything (decoder, optimizer, etc.) but replace model weights with merged SAM state
merged = dict(ckpt)
merged[model_key] = sam.state_dict()
torch.save(merged, OUT_PT)
print("Saved:", OUT_PT)
PY


Decoder-related keys found: ['decoder_state']
Saved: /content/microsam_runs/checkpoints/vit_b_em_organelles/mito_nuc_em_generalist_sam_lora_llrd_r8_d0.95_lr0.0001_of0.1_uf0.1_aug1/mergedbest.pt


# TESTING


LOAD WEIGHTS

In [ ]:
!cp /content/drive/MyDrive/loraFFT/microsam_runs/checkpoints/vit_b_em_organelles/mito_nuc_em_generalist_sam_lora_r8/bestmergedwith_decoder.pt /content/

TESTING LUCCHI

In [ ]:
%%bash
set -e
source /usr/local/bin/activate
conda activate micro_sam3

cat > /content/obtain_lucchi_urocell_kasthuri_vnc_em_datasets.py <<'PY'
import os
import torch
from torch.utils.data import random_split

from torch_em import get_data_loader
from torch_em.data import ConcatDataset, MinInstanceSampler, datasets
from micro_sam.training.util import ResizeRawTrafo, ResizeLabelTrafo


# -----------------------------
# helpers
# -----------------------------
def _to_tt_split(split: str) -> str:
    # map ("val","validation") -> "test" for datasets that only support train/test
    return "test" if split in ("val", "valid", "validation") else split


def _split_dataset(ds, val_fraction=0.1, seed=0):
    n_total = len(ds)
    n_val = max(1, int(val_fraction * n_total))
    n_train = n_total - n_val
    gen = torch.Generator().manual_seed(seed)
    return random_split(ds, [n_train, n_val], generator=gen)


def _common_transforms(patch_shape):
    raw_tf  = ResizeRawTrafo(patch_shape[1:], do_rescaling=False, ensure_rgb=False)
    lab_tf  = ResizeLabelTrafo(patch_shape[1:])
    return raw_tf, lab_tf


# -----------------------------
# LUCCHI
# -----------------------------
def _get_lucchi(input_path, patch_shape, split):
    split = _to_tt_split(split)
    path = os.path.join(input_path, "lucchi")
    raw_tf, lab_tf = _common_transforms(patch_shape)
    return datasets.get_lucchi_dataset(
        path=path,
        patch_shape=patch_shape,
        download=True,
        split=split,
        ndim=2,
        sampler=MinInstanceSampler(min_num_instances=2),
        raw_transform=raw_tf,
        label_transform=lab_tf,
    )


# -----------------------------
# UROCELL (no split support in your torch-em)
# -----------------------------
def _get_urocell_full(input_path, patch_shape, target="mito"):
    path = os.path.join(input_path, "urocell")
    raw_tf, lab_tf = _common_transforms(patch_shape)
    return datasets.get_uro_cell_dataset(
        path=path,
        target=target,
        patch_shape=patch_shape,
        download=True,
        ndim=2,
        sampler=MinInstanceSampler(min_num_instances=2),
        raw_transform=raw_tf,
        label_transform=lab_tf,
    )


# -----------------------------
# KASTHURI
# -----------------------------
def _get_kasthuri(input_path, patch_shape, split):
    split = _to_tt_split(split)
    path = os.path.join(input_path, "kasthuri")
    raw_tf, lab_tf = _common_transforms(patch_shape)
    return datasets.get_kasthuri_dataset(
        path=path,
        split=split,
        patch_shape=patch_shape,
        download=True,
        ndim=2,
        sampler=MinInstanceSampler(min_num_instances=2),
        raw_transform=raw_tf,
        label_transform=lab_tf,
    )


# -----------------------------
# VNC (torch-em name: get_vnc_mito_dataset)
# -----------------------------
def _get_vnc_try_split(input_path, patch_shape, split):
    """
    Try to load VNC with split=... (if supported by this torch-em version).
    If not supported, raise TypeError -> handled by wrapper.
    """
    split = _to_tt_split(split)
    path = os.path.join(input_path, "vnc")
    raw_tf, lab_tf = _common_transforms(patch_shape)
    return datasets.get_vnc_mito_dataset(
        path=path,
        split=split,
        patch_shape=patch_shape,
        download=True,
        ndim=2,  # IMPORTANT: keep 2D so x is [B,1,H,W]
        sampler=MinInstanceSampler(min_num_instances=2),
        raw_transform=raw_tf,
        label_transform=lab_tf,
    )


def _get_vnc_train_val(input_path, patch_shape, val_fraction=0.1, seed=0):
    """
    Returns (train, val) datasets for VNC, robust across torch-em versions.
    """
    try:
        vnc_train = _get_vnc_try_split(input_path, patch_shape, split="train")
        vnc_val   = _get_vnc_try_split(input_path, patch_shape, split="val")
        return vnc_train, vnc_val
    except TypeError:
        # fallback: no split kwarg -> download full dataset once then split ourselves
        path = os.path.join(input_path, "vnc")
        raw_tf, lab_tf = _common_transforms(patch_shape)
        vnc_full = datasets.get_vnc_mito_dataset(
            path=path,
            patch_shape=patch_shape,
            download=True,
            ndim=2,
            sampler=MinInstanceSampler(min_num_instances=2),
            raw_transform=raw_tf,
            label_transform=lab_tf,
        )
        vnc_train, vnc_val = _split_dataset(vnc_full, val_fraction=val_fraction, seed=seed)
        return vnc_train, vnc_val


def _set_sampling_attempts(ds, n=5000):
    # handles Subset wrappers too
    if hasattr(ds, "max_sampling_attempts"):
        ds.max_sampling_attempts = n
    if hasattr(ds, "dataset") and hasattr(ds.dataset, "max_sampling_attempts"):
        ds.dataset.max_sampling_attempts = n


# -----------------------------
# TRAIN LOADERS (generalist)
# -----------------------------
def get_generalist_lucchi_urocell_kasthuri_vnc_loaders(
    input_path,
    patch_shape,
    num_workers=0,
    uro_target="mito",
    uro_val_fraction=0.1,
    vnc_val_fraction=0.1,
    split_seed=0,
    batch_size=1,
):
    lucchi_train = _get_lucchi(input_path, patch_shape, split="train")
    lucchi_val   = _get_lucchi(input_path, patch_shape, split="val")

    uro_full = _get_urocell_full(input_path, patch_shape, target=uro_target)
    uro_train, uro_val = _split_dataset(uro_full, val_fraction=uro_val_fraction, seed=split_seed)

    kas_train = _get_kasthuri(input_path, patch_shape, split="train")
    kas_val   = _get_kasthuri(input_path, patch_shape, split="val")

    vnc_train, vnc_val = _get_vnc_train_val(
        input_path, patch_shape, val_fraction=vnc_val_fraction, seed=split_seed
    )

    train_ds = ConcatDataset(lucchi_train, uro_train, kas_train, vnc_train)
    val_ds   = ConcatDataset(lucchi_val,   uro_val,   kas_val,   vnc_val)

    for ds in [lucchi_train, lucchi_val, kas_train, kas_val, uro_full, vnc_train, vnc_val]:
        _set_sampling_attempts(ds, 5000)

    train_loader = get_data_loader(train_ds, batch_size=batch_size, shuffle=True,  num_workers=num_workers)
    val_loader   = get_data_loader(val_ds,   batch_size=batch_size, shuffle=False, num_workers=num_workers)
    return train_loader, val_loader


# -----------------------------
# EVAL LOADERS
# -----------------------------
def get_eval_loaders(
    input_path,
    patch_shape,
    num_workers=0,
    uro_target="mito",
    uro_val_fraction=0.1,
    vnc_val_fraction=0.1,
    split_seed=0,
    batch_size=1,
):
    # Lucchi test
    lucchi_test = _get_lucchi(input_path, patch_shape, split="test")
    _set_sampling_attempts(lucchi_test, 5000)

    # UroCell val (via random split)
    uro_full = _get_urocell_full(input_path, patch_shape, target=uro_target)
    _, uro_val = _split_dataset(uro_full, val_fraction=uro_val_fraction, seed=split_seed)
    _set_sampling_attempts(uro_full, 5000)

    # Kasthuri test
    kas_test = _get_kasthuri(input_path, patch_shape, split="test")
    _set_sampling_attempts(kas_test, 5000)

    # VNC val/test:
    # we return val loader (like urocell) AND test loader (if split exists it'll be true test, else another val-split)
    vnc_train, vnc_val = _get_vnc_train_val(
        input_path, patch_shape, val_fraction=vnc_val_fraction, seed=split_seed
    )
    _set_sampling_attempts(vnc_train, 5000)
    _set_sampling_attempts(vnc_val, 5000)

    lucchi_test_loader = get_data_loader(lucchi_test, batch_size=batch_size, shuffle=False, num_workers=num_workers)
    urocell_val_loader = get_data_loader(uro_val,     batch_size=batch_size, shuffle=False, num_workers=num_workers)
    kasthuri_test_loader = get_data_loader(kas_test,  batch_size=batch_size, shuffle=False, num_workers=num_workers)
    vnc_val_loader = get_data_loader(vnc_val,         batch_size=batch_size, shuffle=False, num_workers=num_workers)

    return lucchi_test_loader, urocell_val_loader, kasthuri_test_loader, vnc_val_loader
PY



In [ ]:
%%bash
set -e
source /usr/local/bin/activate
conda activate micro_sam3

cat > /content/eval_util.py <<'PY'
import os
from glob import glob

def get_paths(dataset, split="test"):
    root = f"/content/microsam_eval_data/{dataset}/{split}"
    image_paths = sorted(glob(os.path.join(root, "images", "*")))
    gt_paths    = sorted(glob(os.path.join(root, "labels", "*")))
    return image_paths, gt_paths
PY

echo " Wrote /content/eval_util.py"


 Wrote /content/eval_util.py


In [ ]:
%%bash
set -e
source /usr/local/bin/activate
conda activate micro_sam3

python - << 'PY'
import os, sys
import numpy as np
sys.path.append("/content")

from micro_sam.evaluation.inference import run_instance_segmentation_with_decoder
import micro_sam.evaluation.instance_segmentation as instseg
from eval_util import get_paths

# ---------- PATCH micro-sam's writer (THIS is the important part) ----------
try:
    import tifffile
except ImportError:
    tifffile = None

def safe_imwrite(path, arr, **kwargs):
    arr = np.asarray(arr).squeeze()
    # force .tif extension
    base, _ = os.path.splitext(path)
    path = base + ".tif"
    # ignore png-specific kwargs like compression=5
    if tifffile is not None:
        tifffile.imwrite(path, arr)
    else:
        # fallback: use imageio v3 with tifffile plugin if available
        import imageio.v3 as iio
        iio.imwrite(path, arr)
    return path

# patch the exact module used by micro-sam
instseg.imageio.imwrite = safe_imwrite
# -------------------------------------------------------------------------

# -------- CONFIG --------
DATASET = "lucchi"      # or "lucchi"
MODEL_TYPE = "vit_b_em_organelles"     #  must be vit_t/vit_b/vit_l/vit_h
CHECKPOINT = "/content/microsam_runs/checkpoints/vit_b_em_organelles/mito_nuc_em_generalist_sam_lora_llrd_r8_d0.95_lr0.0001_of0.1_uf0.1_aug1/mergedbest.pt"
EXP_DIR = "/content/microsam_eval_runs/lucchi_ais"
# ------------------------

assert os.path.exists(CHECKPOINT), f"Checkpoint not found: {CHECKPOINT}"
os.makedirs(EXP_DIR, exist_ok=True)

val_imgs, val_gts = get_paths(DATASET, split="val")
test_imgs, _ = get_paths(DATASET, split="test")

pred_dir = run_instance_segmentation_with_decoder(
    checkpoint=CHECKPOINT,
    model_type=MODEL_TYPE,
    experiment_folder=EXP_DIR,
    val_image_paths=val_imgs,
    val_gt_paths=val_gts,
    test_image_paths=test_imgs,
)

print(" Predictions written to:", pred_dir)
PY


Best grid-search result: 0.7424919996669996 with parmeters:
 center_distance_threshold = 0.3, boundary_distance_threshold = 0.4, distance_smoothing = 1.8, min_size = 100.0

 Predictions written to: /content/microsam_eval_runs/lucchi_ais/instance_segmentation_with_decoder/inference


Run inference for automatic mask generation: 100%|██████████| 100/100 [00:42<00:00,  2.38it/s]


In [ ]:
%%bash
set -e
source /usr/local/bin/activate
conda activate micro_sam3

python - << 'PY'
import os, sys
from glob import glob
sys.path.append("/content")

from micro_sam.evaluation.evaluation import run_evaluation
from eval_util import get_paths

DATASET = "lucchi"
EXP_DIR = "/content/microsam_eval_runs/lucchi_ais"

#   where micro-sam actually wrote the predictions
pred_dir = os.path.join(EXP_DIR, "instance_segmentation_with_decoder", "inference")

_, gt_paths = get_paths(DATASET, split="test")
pred_paths = sorted(glob(os.path.join(pred_dir, "*.tif")))

print("GT:", len(gt_paths))
print("Pred:", len(pred_paths))
print("Pred dir:", pred_dir)
print("First pred:", pred_paths[0] if pred_paths else None)
print("First gt  :", gt_paths[0] if gt_paths else None)

assert len(gt_paths) == len(pred_paths), "Counts mismatch; filenames might not align."

out_csv = os.path.join(EXP_DIR, "results", "instance_segmentation_with_decoder.csv")
res = run_evaluation(gt_paths, pred_paths, save_path=out_csv)

print("  Saved:", out_csv)
print(res)
PY


GT: 100
Pred: 100
Pred dir: /content/microsam_eval_runs/lucchi_ais/instance_segmentation_with_decoder/inference
First pred: /content/microsam_eval_runs/lucchi_ais/instance_segmentation_with_decoder/inference/lucchi_test_0000.tif
First gt  : /content/microsam_eval_data/lucchi/test/labels/lucchi_test_0000.png
  Saved: /content/microsam_eval_runs/lucchi_ais/results/instance_segmentation_with_decoder.csv
        mSA      SA50     SA75  Precision    Recall  F1 Score
0  0.721712  0.839101  0.81027   0.937144  0.887775  0.904741


Evaluate predictions: 100%|██████████| 100/100 [00:01<00:00, 56.51it/s]


VNC

In [ ]:
%%bash
set -e
source /usr/local/bin/activate
conda activate micro_sam3

python - << 'PY'
import os, sys
import numpy as np
import imageio.v2 as imageio
sys.path.append("/content")

from obtain_lucchi_urocell_kasthuri_vnc_em_datasets import get_eval_loaders

# -------- CONFIG --------
DATASET = "vnc"   # "lucchi" or "urocell" or "kasthuri" or "vnc"
INST_CH = 0
N_VAL  = 50
N_TEST = 100
OUT_ROOT = "/content/microsam_eval_data"
PATCH = (1, 512, 512)
# ------------------------

lucchi_test_loader, urocell_val_loader, kasthuri_test_loader, vnc_val_loader = get_eval_loaders(
    input_path="/content/em_data",
    patch_shape=PATCH,
    num_workers=0,
    uro_target="mito",
    uro_val_fraction=0.1,
    vnc_val_fraction=0.1,
    split_seed=0,
    batch_size=1
)

# choose loader
if DATASET == "lucchi":
    loader = lucchi_test_loader
elif DATASET == "urocell":
    loader = urocell_val_loader
elif DATASET == "kasthuri":
    loader = kasthuri_test_loader
elif DATASET == "vnc":
    loader = vnc_val_loader
else:
    raise ValueError("DATASET must be one of: lucchi, urocell, kasthuri, vnc")

def ensure_dirs(base):
    for s in ("val", "test"):
        os.makedirs(os.path.join(base, s, "images"), exist_ok=True)
        os.makedirs(os.path.join(base, s, "labels"), exist_ok=True)

def to_uint8(x):
    x = x.astype(np.float32)
    x = (x - x.min()) / (x.max() - x.min() + 1e-8)
    return (255 * x).astype(np.uint8)

def to_2d(a):
    a = np.asarray(a)
    a = np.squeeze(a)
    # if still >2D, pick a reasonable slice/channel
    while a.ndim > 2:
        a = a[0]
    if a.ndim != 2:
        raise ValueError(f"Expected 2D, got {a.shape}")
    return a

base = os.path.join(OUT_ROOT, DATASET)
ensure_dirs(base)

def dump(split, n):
    img_dir = os.path.join(base, split, "images")
    lab_dir = os.path.join(base, split, "labels")
    for i, (x, y) in enumerate(loader):
        if i >= n:
            break

        # x: ensure 2D
        img = x[0].numpy()        # robust across datasets
        img = to_2d(img)
        img = to_uint8(img)

        # y: instance ids channel -> ensure 2D
        inst = y[0, INST_CH].numpy()
        inst = to_2d(inst).astype(np.uint16)

        imageio.imwrite(f"{img_dir}/{DATASET}_{split}_{i:04d}.png", img)
        imageio.imwrite(f"{lab_dir}/{DATASET}_{split}_{i:04d}.png", inst)

dump("val",  N_VAL)
dump("test", N_TEST)

print("Exported evaluation data to:", base)
print("val images :", len(os.listdir(os.path.join(base,"val","images"))))
print("test images:", len(os.listdir(os.path.join(base,"test","images"))))
PY


Exported evaluation data to: /content/microsam_eval_data/vnc
val images : 8
test images: 8


In [ ]:
%%bash
set -e
source /usr/local/bin/activate
conda activate micro_sam3

cat > /content/eval_util.py <<'PY'
import os
from glob import glob

def get_paths(dataset, split="test"):
    root = f"/content/microsam_eval_data/{dataset}/{split}"
    image_paths = sorted(glob(os.path.join(root, "images", "*")))
    gt_paths    = sorted(glob(os.path.join(root, "labels", "*")))
    return image_paths, gt_paths
PY

echo " Wrote /content/eval_util.py"


 Wrote /content/eval_util.py


In [ ]:
%%bash
set -e
source /usr/local/bin/activate
conda activate micro_sam3

python - << 'PY'
import os, sys
import numpy as np
sys.path.append("/content")

from micro_sam.evaluation.inference import run_instance_segmentation_with_decoder
import micro_sam.evaluation.instance_segmentation as instseg
from eval_util import get_paths

# ---------- PATCH micro-sam's writer (THIS is the important part) ----------
try:
    import tifffile
except ImportError:
    tifffile = None

def safe_imwrite(path, arr, **kwargs):
    arr = np.asarray(arr).squeeze()
    # force .tif extension
    base, _ = os.path.splitext(path)
    path = base + ".tif"
    # ignore png-specific kwargs like compression=5
    if tifffile is not None:
        tifffile.imwrite(path, arr)
    else:
        # fallback: use imageio v3 with tifffile plugin if available
        import imageio.v3 as iio
        iio.imwrite(path, arr)
    return path

# patch the exact module used by micro-sam
instseg.imageio.imwrite = safe_imwrite
# -------------------------------------------------------------------------

# -------- CONFIG --------
DATASET = "vnc"      # or "lucchi"
MODEL_TYPE = "vit_b_em_orgenelles"     #  must be vit_t/vit_b/vit_l/vit_h
CHECKPOINT = "/content/microsam_runs/checkpoints/vit_b_em_organelles/mito_nuc_em_generalist_sam_lora_llrd_r8_d0.95_lr0.0001_of0.1_uf0.1_aug1/mergedbest.pt"
EXP_DIR = "/content/microsam_eval_runs/vnc_ais"
# ------------------------

assert os.path.exists(CHECKPOINT), f"Checkpoint not found: {CHECKPOINT}"
os.makedirs(EXP_DIR, exist_ok=True)

val_imgs, val_gts = get_paths(DATASET, split="val")
test_imgs, _ = get_paths(DATASET, split="test")

pred_dir = run_instance_segmentation_with_decoder(
    checkpoint=CHECKPOINT,
    model_type=MODEL_TYPE,
    experiment_folder=EXP_DIR,
    val_image_paths=val_imgs,
    val_gt_paths=val_gts,
    test_image_paths=test_imgs,
)

print(" Predictions written to:", pred_dir)
PY


Best grid-search result: 0.5225536708890044 with parmeters:
 center_distance_threshold = 0.3, boundary_distance_threshold = 0.3, distance_smoothing = 1.0, min_size = 50.0

 Predictions written to: /content/microsam_eval_runs/vnc_ais/instance_segmentation_with_decoder/inference


Run inference for automatic mask generation: 100%|██████████| 8/8 [00:03<00:00,  2.40it/s]


In [ ]:
%%bash
set -e
source /usr/local/bin/activate
conda activate micro_sam3

python - << 'PY'
import os, sys
from glob import glob
sys.path.append("/content")

from micro_sam.evaluation.evaluation import run_evaluation
from eval_util import get_paths

DATASET = "vnc"
EXP_DIR = "/content/microsam_eval_runs/vnc_ais"

#   where micro-sam actually wrote the predictions
pred_dir = os.path.join(EXP_DIR, "instance_segmentation_with_decoder", "inference")

_, gt_paths = get_paths(DATASET, split="test")
pred_paths = sorted(glob(os.path.join(pred_dir, "*.tif")))

print("GT:", len(gt_paths))
print("Pred:", len(pred_paths))
print("Pred dir:", pred_dir)
print("First pred:", pred_paths[0] if pred_paths else None)
print("First gt  :", gt_paths[0] if gt_paths else None)

assert len(gt_paths) == len(pred_paths), "Counts mismatch; filenames might not align."

out_csv = os.path.join(EXP_DIR, "results", "instance_segmentation_with_decoder.csv")
res = run_evaluation(gt_paths, pred_paths, save_path=out_csv)

print("  Saved:", out_csv)
print(res)
PY


GT: 8
Pred: 8
Pred dir: /content/microsam_eval_runs/vnc_ais/instance_segmentation_with_decoder/inference
First pred: /content/microsam_eval_runs/vnc_ais/instance_segmentation_with_decoder/inference/vnc_test_0000.tif
First gt  : /content/microsam_eval_data/vnc/test/labels/vnc_test_0000.png
  Saved: /content/microsam_eval_runs/vnc_ais/results/instance_segmentation_with_decoder.csv
        mSA      SA50      SA75  Precision    Recall  F1 Score
0  0.699331  0.850758  0.800758        1.0  0.850758   0.89674


Evaluate predictions: 100%|██████████| 8/8 [00:00<00:00, 53.85it/s]


KASTURI TESTING

In [ ]:
%%bash
set -e
source /usr/local/bin/activate
conda activate micro_sam3

python - << 'PY'
import os, sys
import numpy as np
import imageio.v2 as imageio
sys.path.append("/content")

from obtain_lucchi_urocell_kasthuri_vnc_em_datasets import get_eval_loaders

# -------- CONFIG --------
DATASET = "kasthuri"   # "lucchi" or "urocell" or "kasthuri" or "vnc"
INST_CH = 0
N_VAL  = 50
N_TEST = 100
OUT_ROOT = "/content/microsam_eval_data"
PATCH = (1, 512, 512)
# ------------------------

lucchi_test_loader, urocell_val_loader, kasthuri_test_loader, vnc_val_loader = get_eval_loaders(
    input_path="/content/em_data",
    patch_shape=PATCH,
    num_workers=0,
    uro_target="mito",
    uro_val_fraction=0.1,
    vnc_val_fraction=0.1,
    split_seed=0,
    batch_size=1
)

# choose loader
if DATASET == "lucchi":
    loader = lucchi_test_loader
elif DATASET == "urocell":
    loader = urocell_val_loader
elif DATASET == "kasthuri":
    loader = kasthuri_test_loader
elif DATASET == "vnc":
    loader = vnc_val_loader
else:
    raise ValueError("DATASET must be one of: lucchi, urocell, kasthuri, vnc")

def ensure_dirs(base):
    for s in ("val", "test"):
        os.makedirs(os.path.join(base, s, "images"), exist_ok=True)
        os.makedirs(os.path.join(base, s, "labels"), exist_ok=True)

def to_uint8(x):
    x = x.astype(np.float32)
    x = (x - x.min()) / (x.max() - x.min() + 1e-8)
    return (255 * x).astype(np.uint8)

def to_2d(a):
    a = np.asarray(a)
    a = np.squeeze(a)
    # if still >2D, pick a reasonable slice/channel
    while a.ndim > 2:
        a = a[0]
    if a.ndim != 2:
        raise ValueError(f"Expected 2D, got {a.shape}")
    return a

base = os.path.join(OUT_ROOT, DATASET)
ensure_dirs(base)

def dump(split, n):
    img_dir = os.path.join(base, split, "images")
    lab_dir = os.path.join(base, split, "labels")
    for i, (x, y) in enumerate(loader):
        if i >= n:
            break

        # x: ensure 2D
        img = x[0].numpy()        # robust across datasets
        img = to_2d(img)
        img = to_uint8(img)

        # y: instance ids channel -> ensure 2D
        inst = y[0, INST_CH].numpy()
        inst = to_2d(inst).astype(np.uint16)

        imageio.imwrite(f"{img_dir}/{DATASET}_{split}_{i:04d}.png", img)
        imageio.imwrite(f"{lab_dir}/{DATASET}_{split}_{i:04d}.png", inst)

dump("val",  N_VAL)
dump("test", N_TEST)

print("Exported evaluation data to:", base)
print("val images :", len(os.listdir(os.path.join(base,"val","images"))))
print("test images:", len(os.listdir(os.path.join(base,"test","images"))))
PY


Exported evaluation data to: /content/microsam_eval_data/kasthuri
val images : 50
test images: 100


In [ ]:
%%bash
set -e
source /usr/local/bin/activate
conda activate micro_sam3

cat > /content/eval_util.py <<'PY'
import os
from glob import glob

def get_paths(dataset, split="test"):
    root = f"/content/microsam_eval_data/{dataset}/{split}"
    image_paths = sorted(glob(os.path.join(root, "images", "*")))
    gt_paths    = sorted(glob(os.path.join(root, "labels", "*")))
    return image_paths, gt_paths
PY

echo " Wrote /content/eval_util.py"


 Wrote /content/eval_util.py


In [ ]:
%%bash
set -e
source /usr/local/bin/activate
conda activate micro_sam3

python - << 'PY'
import os, sys
import numpy as np
sys.path.append("/content")

from micro_sam.evaluation.inference import run_instance_segmentation_with_decoder
import micro_sam.evaluation.instance_segmentation as instseg
from eval_util import get_paths

# ---------- PATCH micro-sam's writer (THIS is the important part) ----------
try:
    import tifffile
except ImportError:
    tifffile = None

def safe_imwrite(path, arr, **kwargs):
    arr = np.asarray(arr).squeeze()
    # force .tif extension
    base, _ = os.path.splitext(path)
    path = base + ".tif"
    # ignore png-specific kwargs like compression=5
    if tifffile is not None:
        tifffile.imwrite(path, arr)
    else:
        # fallback: use imageio v3 with tifffile plugin if available
        import imageio.v3 as iio
        iio.imwrite(path, arr)
    return path

# patch the exact module used by micro-sam
instseg.imageio.imwrite = safe_imwrite
# -------------------------------------------------------------------------

# -------- CONFIG --------
DATASET = "kasthuri"      # or "lucchi"
MODEL_TYPE = "vit_b_em_orgenelles"     #  must be vit_t/vit_b/vit_l/vit_h
CHECKPOINT = "/content/microsam_runs/checkpoints/vit_b_em_organelles/mito_nuc_em_generalist_sam_lora_llrd_r8_d0.95_lr0.0001_of0.1_uf0.1_aug1/mergedbest.pt"
EXP_DIR = "/content/microsam_eval_runs/kasthuri_ais"
# ------------------------

assert os.path.exists(CHECKPOINT), f"Checkpoint not found: {CHECKPOINT}"
os.makedirs(EXP_DIR, exist_ok=True)

val_imgs, val_gts = get_paths(DATASET, split="val")
test_imgs, _ = get_paths(DATASET, split="test")

pred_dir = run_instance_segmentation_with_decoder(
    checkpoint=CHECKPOINT,
    model_type=MODEL_TYPE,
    experiment_folder=EXP_DIR,
    val_image_paths=val_imgs,
    val_gt_paths=val_gts,
    test_image_paths=test_imgs,
)

print(" Predictions written to:", pred_dir)
PY


Best grid-search result: 0.7579420539213109 with parmeters:
 center_distance_threshold = 0.4, boundary_distance_threshold = 0.7, distance_smoothing = 1.8, min_size = 50.0

 Predictions written to: /content/microsam_eval_runs/kasthuri_ais/instance_segmentation_with_decoder/inference


Run inference for automatic mask generation: 100%|██████████| 100/100 [00:42<00:00,  2.33it/s]


In [ ]:
%%bash
set -e
source /usr/local/bin/activate
conda activate micro_sam3

python - << 'PY'
import os, sys
from glob import glob
sys.path.append("/content")

from micro_sam.evaluation.evaluation import run_evaluation
from eval_util import get_paths

DATASET = "kasthuri"
EXP_DIR = "/content/microsam_eval_runs/kasthuri_ais"

#   where micro-sam actually wrote the predictions
pred_dir = os.path.join(EXP_DIR, "instance_segmentation_with_decoder", "inference")

_, gt_paths = get_paths(DATASET, split="test")
pred_paths = sorted(glob(os.path.join(pred_dir, "*.tif")))

print("GT:", len(gt_paths))
print("Pred:", len(pred_paths))
print("Pred dir:", pred_dir)
print("First pred:", pred_paths[0] if pred_paths else None)
print("First gt  :", gt_paths[0] if gt_paths else None)

assert len(gt_paths) == len(pred_paths), "Counts mismatch; filenames might not align."

out_csv = os.path.join(EXP_DIR, "results", "instance_segmentation_with_decoder.csv")
res = run_evaluation(gt_paths, pred_paths, save_path=out_csv)

print("  Saved:", out_csv)
print(res)
PY


GT: 100
Pred: 100
Pred dir: /content/microsam_eval_runs/kasthuri_ais/instance_segmentation_with_decoder/inference
First pred: /content/microsam_eval_runs/kasthuri_ais/instance_segmentation_with_decoder/inference/kasthuri_test_0000.tif
First gt  : /content/microsam_eval_data/kasthuri/test/labels/kasthuri_test_0000.png
  Saved: /content/microsam_eval_runs/kasthuri_ais/results/instance_segmentation_with_decoder.csv
        mSA      SA50      SA75  Precision    Recall  F1 Score
0  0.729055  0.846209  0.780393   0.940298  0.894141  0.908366


Evaluate predictions: 100%|██████████| 100/100 [00:01<00:00, 52.91it/s]


UROCEL TESTING

In [ ]:
%%bash
set -e
source /usr/local/bin/activate
conda activate micro_sam3

python - << 'PY'
import os, sys
import numpy as np
import imageio.v2 as imageio
sys.path.append("/content")

from obtain_lucchi_urocell_kasthuri_vnc_em_datasets import get_eval_loaders

# -------- CONFIG --------
DATASET = "urocell"   # "lucchi" or "urocell" or "kasthuri" or "vnc"
INST_CH = 0
N_VAL  = 50
N_TEST = 100
OUT_ROOT = "/content/microsam_eval_data"
PATCH = (1, 512, 512)
# ------------------------

lucchi_test_loader, urocell_val_loader, kasthuri_test_loader, vnc_val_loader = get_eval_loaders(
    input_path="/content/em_data",
    patch_shape=PATCH,
    num_workers=0,
    uro_target="mito",
    uro_val_fraction=0.1,
    vnc_val_fraction=0.1,
    split_seed=0,
    batch_size=1
)

# choose loader
if DATASET == "lucchi":
    loader = lucchi_test_loader
elif DATASET == "urocell":
    loader = urocell_val_loader
elif DATASET == "kasthuri":
    loader = kasthuri_test_loader
elif DATASET == "vnc":
    loader = vnc_val_loader
else:
    raise ValueError("DATASET must be one of: lucchi, urocell, kasthuri, vnc")

def ensure_dirs(base):
    for s in ("val", "test"):
        os.makedirs(os.path.join(base, s, "images"), exist_ok=True)
        os.makedirs(os.path.join(base, s, "labels"), exist_ok=True)

def to_uint8(x):
    x = x.astype(np.float32)
    x = (x - x.min()) / (x.max() - x.min() + 1e-8)
    return (255 * x).astype(np.uint8)

def to_2d(a):
    a = np.asarray(a)
    a = np.squeeze(a)
    # if still >2D, pick a reasonable slice/channel
    while a.ndim > 2:
        a = a[0]
    if a.ndim != 2:
        raise ValueError(f"Expected 2D, got {a.shape}")
    return a

base = os.path.join(OUT_ROOT, DATASET)
ensure_dirs(base)

def dump(split, n):
    img_dir = os.path.join(base, split, "images")
    lab_dir = os.path.join(base, split, "labels")
    for i, (x, y) in enumerate(loader):
        if i >= n:
            break

        # x: ensure 2D
        img = x[0].numpy()        # robust across datasets
        img = to_2d(img)
        img = to_uint8(img)

        # y: instance ids channel -> ensure 2D
        inst = y[0, INST_CH].numpy()
        inst = to_2d(inst).astype(np.uint16)

        imageio.imwrite(f"{img_dir}/{DATASET}_{split}_{i:04d}.png", img)
        imageio.imwrite(f"{lab_dir}/{DATASET}_{split}_{i:04d}.png", inst)

dump("val",  N_VAL)
dump("test", N_TEST)

print("Exported evaluation data to:", base)
print("val images :", len(os.listdir(os.path.join(base,"val","images"))))
print("test images:", len(os.listdir(os.path.join(base,"test","images"))))
PY


Exported evaluation data to: /content/microsam_eval_data/urocell
val images : 32
test images: 32


In [ ]:
%%bash
set -e
source /usr/local/bin/activate
conda activate micro_sam3

cat > /content/eval_util.py <<'PY'
import os
from glob import glob

def get_paths(dataset, split="test"):
    root = f"/content/microsam_eval_data/{dataset}/{split}"
    image_paths = sorted(glob(os.path.join(root, "images", "*")))
    gt_paths    = sorted(glob(os.path.join(root, "labels", "*")))
    return image_paths, gt_paths
PY

echo " Wrote /content/eval_util.py"


 Wrote /content/eval_util.py


In [ ]:
%%bash
set -e
source /usr/local/bin/activate
conda activate micro_sam3

python - << 'PY'
import os, sys
import numpy as np
sys.path.append("/content")

from micro_sam.evaluation.inference import run_instance_segmentation_with_decoder
import micro_sam.evaluation.instance_segmentation as instseg
from eval_util import get_paths

# ---------- PATCH micro-sam's writer (THIS is the important part) ----------
try:
    import tifffile
except ImportError:
    tifffile = None

def safe_imwrite(path, arr, **kwargs):
    arr = np.asarray(arr).squeeze()
    # force .tif extension
    base, _ = os.path.splitext(path)
    path = base + ".tif"
    # ignore png-specific kwargs like compression=5
    if tifffile is not None:
        tifffile.imwrite(path, arr)
    else:
        # fallback: use imageio v3 with tifffile plugin if available
        import imageio.v3 as iio
        iio.imwrite(path, arr)
    return path

# patch the exact module used by micro-sam
instseg.imageio.imwrite = safe_imwrite
# -------------------------------------------------------------------------

# -------- CONFIG --------
DATASET = "urocell"      # or "lucchi"
MODEL_TYPE = "vit_b_em_orgenelles"     #  must be vit_t/vit_b/vit_l/vit_h
CHECKPOINT = "/content/microsam_runs/checkpoints/vit_b_em_organelles/mito_nuc_em_generalist_sam_lora_llrd_r8_d0.95_lr0.0001_of0.1_uf0.1_aug1/mergedbest.pt"
EXP_DIR = "/content/microsam_eval_runs/urocell_ais"
# ------------------------

assert os.path.exists(CHECKPOINT), f"Checkpoint not found: {CHECKPOINT}"
os.makedirs(EXP_DIR, exist_ok=True)

val_imgs, val_gts = get_paths(DATASET, split="val")
test_imgs, _ = get_paths(DATASET, split="test")

pred_dir = run_instance_segmentation_with_decoder(
    checkpoint=CHECKPOINT,
    model_type=MODEL_TYPE,
    experiment_folder=EXP_DIR,
    val_image_paths=val_imgs,
    val_gt_paths=val_gts,
    test_image_paths=test_imgs,
)

print(" Predictions written to:", pred_dir)
PY


Best grid-search result: 0.6616527518547785 with parmeters:
 center_distance_threshold = 0.5, boundary_distance_threshold = 0.4, distance_smoothing = 1.0, min_size = 50.0

 Predictions written to: /content/microsam_eval_runs/urocell_ais/instance_segmentation_with_decoder/inference


Run inference for automatic mask generation: 100%|██████████| 32/32 [00:13<00:00,  2.44it/s]


In [ ]:
%%bash
set -e
source /usr/local/bin/activate
conda activate micro_sam3

python - << 'PY'
import os, sys
from glob import glob
sys.path.append("/content")

from micro_sam.evaluation.evaluation import run_evaluation
from eval_util import get_paths

DATASET = "urocell"
EXP_DIR = "/content/microsam_eval_runs/urocell_ais"

#   where micro-sam actually wrote the predictions
pred_dir = os.path.join(EXP_DIR, "instance_segmentation_with_decoder", "inference")

_, gt_paths = get_paths(DATASET, split="test")
pred_paths = sorted(glob(os.path.join(pred_dir, "*.tif")))

print("GT:", len(gt_paths))
print("Pred:", len(pred_paths))
print("Pred dir:", pred_dir)
print("First pred:", pred_paths[0] if pred_paths else None)
print("First gt  :", gt_paths[0] if gt_paths else None)

assert len(gt_paths) == len(pred_paths), "Counts mismatch; filenames might not align."

out_csv = os.path.join(EXP_DIR, "results", "instance_segmentation_with_decoder.csv")
res = run_evaluation(gt_paths, pred_paths, save_path=out_csv)

print("  Saved:", out_csv)
print(res)
PY


GT: 32
Pred: 32
Pred dir: /content/microsam_eval_runs/urocell_ais/instance_segmentation_with_decoder/inference
First pred: /content/microsam_eval_runs/urocell_ais/instance_segmentation_with_decoder/inference/urocell_test_0000.tif
First gt  : /content/microsam_eval_data/urocell/test/labels/urocell_test_0000.png
  Saved: /content/microsam_eval_runs/urocell_ais/results/instance_segmentation_with_decoder.csv
        mSA      SA50      SA75  Precision   Recall  F1 Score
0  0.652084  0.797115  0.742512   0.902975  0.86632  0.874901


Evaluate predictions: 100%|██████████| 32/32 [00:00<00:00, 55.67it/s]
